# 🧠 Hyperparameters and Model Validation

For the types of ML problems (regression, classification, clustering, association rules) see `05.01`.

---

## 1. Model Validation Basics (Supervised Learning)

**Goal:** compare the model's predictions with the real labels (ground truth) on data the model has **not** seen during training.

1. Train the model on labelled training data.
2. Predict the target for held-out data.
3. Compare predictions with the actual labels using a metric.

---

## 2. Analytics Pipeline with Validation

| Step | What happens |
| --- | --- |
| Data preparation | Load labelled data; build the feature matrix `X` and target `y` |
| Model & hyperparameter selection | Choose the method (regression, decision tree, …) and its hyperparameters (tree depth, number of clusters, …) |
| Training | Fit the model on the training data |
| Model display | Inspect the model (regression equation, tree plot, …) |
| **Validation** | Predict on held-out labelled data and compute metrics |
| Application | Apply the model to new, unlabelled data |

> **Parameters vs hyperparameters:** parameters (weights, split thresholds) are *learned* by `.fit()`; hyperparameters (`max_depth`, `n_clusters`, learning rate) are *chosen by you* before training.

## 3. Regression Metrics 📈

- **MAE (Mean Absolute Error)** — average size of the errors, ignoring direction; in the target's units.
  $$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|$$
  🔻 Lower is better.

- **MAPE (Mean Absolute Percentage Error)** — error as a percentage of the actual value. Undefined when any $y_i = 0$ and inflated when actual values are small.
  $$\text{MAPE} = \frac{100\%}{n} \sum_{i=1}^{n} \left| \frac{y_i - \hat{y}_i}{y_i} \right|$$
  🔻 Lower is better.

- **RMSE (Root Mean Squared Error)** — squares errors before averaging, so large errors weigh more than in MAE; in the target's units.
  $$\text{RMSE} = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2 }$$
  🔻 Lower is better.

- **R² (coefficient of determination)** — share of the target's variance explained by the model.
  $$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$
  🔺 Higher is better; 1 is a perfect fit, 0 is no better than predicting the mean, and it can be negative on test data.

## 4. Classification Metrics 📊

Based on the confusion-matrix counts TP, TN, FP, FN (see `08-some-notes` §6 for when to use which).

- **Accuracy** — share of all predictions that are correct. Misleading on imbalanced classes.
  $$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$

- **Precision** — of all predicted positives, how many are truly positive? (fewer false positives)
  $$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

- **Recall (sensitivity)** — of all actual positives, how many did the model find? (fewer false negatives)
  $$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

- **F1-score** — harmonic mean of precision and recall; balances the two.
  $$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

- **ROC curve / AUC** — the ROC curve plots the true positive rate against the false positive rate across all decision thresholds; AUC summarises it in one number (0.5 = random guessing, 1 = perfect separation).

🔺 For all classification metrics above, higher is better (range 0–1).

## 5. Train–Test Split

**Why split?** A model scored on its own training data looks better than it really is. Scoring on held-out data shows how well it **generalises** and exposes overfitting.

Common ratios: **70/30** or **80/20** (train/test). For classification, use `stratify=y` to keep the class proportions equal in both parts.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

---

## 6. Cross-Validation

Split the data into $k$ folds (often 5 or 10). Train on $k-1$ folds, test on the remaining one, and repeat so every fold is the test set once. Report the mean (and spread) of the $k$ scores.

**Benefits:**
- Every observation is used for both training and testing.
- The score depends less on one lucky or unlucky split (lower variance than a single split).

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5, scoring="f1")
print(scores.mean(), scores.std())
```

---

## 7. Underfitting vs Overfitting

| | Training score | Test score | Cause | Typical fix |
| --- | --- | --- | --- | --- |
| **Underfitting** | Poor | Poor | Model too simple | More features, more complex model |
| **Good fit** | Good | Good (close to training) | — | — |
| **Overfitting** | Excellent | Poor | Model memorises noise | Simpler model, regularisation, more data |

---

## 8. Model Selection and Hyperparameter Tuning

Use **three** roles for the data so the final score is not biased by the tuning itself:

| Split | Used to |
| --- | --- |
| **Train** | Fit models with different hyperparameters |
| **Validation** (or cross-validation on the training set) | Pick the best hyperparameters |
| **Test** | Final, one-time evaluation on unseen data |

---

## 9. Grid Search

Systematically tries combinations of hyperparameter values and keeps the best by cross-validated score.

| | `GridSearchCV` | `RandomizedSearchCV` |
| --- | --- | --- |
| **Tries** | Every combination | A fixed number (`n_iter`) of random combinations |
| **Best for** | Small grids | Large grids or continuous ranges (much faster) |

```python
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {"max_depth": [2, 3, 5, 10], "min_samples_leaf": [1, 5, 10]}
search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
search.fit(X_train, y_train)

print(search.best_params_, search.best_score_)
print(search.score(X_test, y_test))  # final check on the untouched test set
```

---

## 🔚 Conclusion
Validation checks that a model generalises to new data. Train–test splits, cross-validation and grid search together let you pick and tune a model without fooling yourself with an optimistic score.